# Paso 1 - Caracterizar el fondo (ruido de reposo)

**KPCL0034 (comedero, "Bandida") + KPCL0035 (bebedero)** - variable `peso`,
senial cruda, **sin filtrar ni eliminar nada** salvo un artefacto de escritura
duplicada ya confirmado (ver celda de documentacion mas abajo).

```
PASO 1 - CARACTERIZACION DEL FONDO
|
+-- 1. Integridad      (NaN, duplicados, timestamps, valores imposibles)
+-- 2. Nivel           (mediana, percentiles, MAD, STD)
+-- 3. Resolucion      (valores enteros, valores unicos, escalones minimos)
+-- 4. Dinamica        (delta_peso, |delta_peso|, direccion, frecuencia)
+-- 5. Temporal        (delta_t, gaps, frecuencia de muestreo)
+-- 6. Velocidad       (delta_peso / delta_t)
+-- 7. Persistencia    (duracion delta_peso=0, estabilidad tolerante)
+-- 8. Comportamiento temporal  (autocorrelacion, por hora del dia)
```

**Dos niveles de agrupamiento, cada uno con su proposito - no se mezclan:**

- `device_id` (UUID crudo): KPCL0034 tiene 2 UUID historicos (abril / mayo-en-
  adelante). Los `diff()` de peso y tiempo se calculan agrupados por **este**
  nivel.
- `device_code` (KPCL0034 / KPCL0035): los 2 UUID de KPCL0034 se **unen como
  un solo dispositivo logico** para todo el reporte/comparacion. KPCL0035
  (bebedero) nunca se mezcla con KPCL0034 en un numero "global".

Notebook dividido en 3 bloques de codigo: (1) carga + integridad + dedup,
(2) nivel + resolucion + dinamica + temporal, (3) velocidad + persistencia +
comportamiento temporal + resumen.

Todavia solo se observa y caracteriza - nada de esto se convierte en regla de
deteccion en este paso.

**Cache (2026-08-29):** este notebook escribe, al final de su celda de carga,
`data/lecturas_limpias.csv` (post-dedup: device_id, device_code, ts, peso) --
los notebooks 02/03/04 leen ese cache en vez de reparsear los CSV crudos.
**Correr este notebook primero.**


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# --- Rutas -------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent  # Investigacion_v2 -> Investigacion -> raiz repo
RAW_DATA_DIR = REPO_ROOT / "11_Data" / "2026"
READINGS_CSV = RAW_DATA_DIR / "readings.csv"
READINGS_ROWS_CSV = RAW_DATA_DIR / "readings_rows.csv"

# device_id (UUID) -> device_code (nombre logico). KPCL0034 tiene 2 UUID
# historicos que son EL MISMO dispositivo fisico. KPCL0035 confirmado via
# Supabase devices (2026-08-29): id=0dc601c0..., device_type="water_bowl".
DEVICE_PROFILES = {
    "KPCL0034": {
        "9510a455-b0e9-4932-8be1-03976d31228a",  # Abril 2026
        "3a460074-e7c3-41bf-ae5a-a011445f927a",  # Mayo 2026 en adelante
    },
    "KPCL0035": {
        "0dc601c0-1533-40c5-b606-6d89eb2d4042",
    },
}
UUID_TO_CODE = {uuid: code for code, uuids in DEVICE_PROFILES.items() for uuid in uuids}
ALL_UUIDS = set(UUID_TO_CODE)
GAP_CUTOFF_S = 300  # 5 min -- mismo umbral ya validado en Gamma/Delta para separar sesiones

# --- Carga: senial cruda, sin resamplear -------------------------------------
USECOLS = ["device_id", "ingested_at", "weight_grams"]

frames = []
for path in (READINGS_CSV, READINGS_ROWS_CSV):
    df_src = pd.read_csv(path, usecols=USECOLS, low_memory=False)
    frames.append(df_src[df_src["device_id"].isin(ALL_UUIDS)])

df = pd.concat(frames, ignore_index=True)
df["device_id"] = df["device_id"].astype("category")
df["device_code"] = df["device_id"].map(UUID_TO_CODE).astype("category")
df["ts"] = pd.to_datetime(df["ingested_at"], format="ISO8601", utc=True)
df["peso"] = pd.to_numeric(df["weight_grams"], errors="coerce", downcast="float")
df = df.drop(columns=["ingested_at", "weight_grams"])
# Ordenar por device_code (no device_id): alfabeticamente "3a460074..." (mayo)
# va antes que "9510a455..." (abril) -- ordenar por el UUID crudo dejaria las
# filas de KPCL0034 en mayo->agosto seguido de abril, NO cronologico.
df = df.sort_values(["device_code", "ts"]).reset_index(drop=True)

print(f"Lecturas totales (antes de dedup): {len(df):,}")
print(df["device_code"].value_counts())

# === 1. INTEGRIDAD ============================================================
n_nan = df["peso"].isna().sum()
print(f"NaN en peso: {n_nan:,} ({n_nan / len(df) * 100:.2f}%)")

dup_device_ts = df.duplicated(["device_id", "ts"]).sum()
print(f"Duplicados exactos (device_id, ts): {dup_device_ts:,}")

orden_ok = df.groupby("device_code", observed=True)["ts"].apply(lambda s: s.is_monotonic_increasing)
print("Timestamps en orden creciente por device_code:")
print(orden_ok)

n_negativos = (df["peso"] < 0).sum()
print(f"Lecturas con peso < 0: {n_negativos:,}")

# --- Fix critico: diff() agrupado por device_id (UUID), no por device_code --
df["delta_peso"] = df.groupby("device_id", observed=True)["peso"].diff()
df["delta_t"] = df.groupby("device_id", observed=True)["ts"].diff().dt.total_seconds()

# --- Hallazgo y correccion: filas duplicadas por escritura doble en abril ----
# Detectado 2026-08-29: 49.6% de las filas del UUID de abril (9510a455) son un
# segundo registro casi instantaneo (~0.2-0.25s despues) del mismo evento, con
# el mismo peso -- NO es senial de alta frecuencia real, es readings.csv
# escribiendo cada muestra 2 veces durante ese periodo (mayo+ y KPCL0035 no lo
# hacen: fraccion de intervalos <1s ahi es ~0.006%). Ver celda de markdown
# siguiente para el detalle completo del hallazgo y la decision.
#
# Criterio de dedup, estricto y mecanico (no un filtro arbitrario):
#   delta_t < 1s  Y  delta_peso == 0 exacto
# Los pares con delta_t<1s pero peso DISTINTO se dejan sin tocar -- podrian
# ser un evento real que coincidio justo con el instante del duplicado.
candidatos_par = df["delta_t"] < 1.0
# notna() explicito: delta_peso NaN (peso faltante) tambien cumple "!= 0" en
# pandas y contaminaria el conteo con casos que no son un mismatch real.
pares_peso_distinto = candidatos_par & df["delta_peso"].notna() & (df["delta_peso"] != 0)
pares_peso_nan = candidatos_par & df["delta_peso"].isna()
print(f"Pares con delta_t<1s y peso distinto de verdad (sin tocar): {pares_peso_distinto.sum()}")
print(f"Pares con delta_t<1s pero peso NaN (no es mismatch, se ignoran): {pares_peso_nan.sum()}")
if pares_peso_distinto.sum() > 0:
    print(df.loc[pares_peso_distinto, ["device_code", "ts", "peso", "delta_peso"]])

# marca la SEGUNDA fila de cada par (la que tiene delta_t chico contra la
# anterior) -- esa es la que se elimina, se conserva la primera del par.
es_dup_par = candidatos_par & (df["delta_peso"] == 0)
n_abril_antes = (df["device_id"] == "9510a455-b0e9-4932-8be1-03976d31228a").sum()
n_kpcl0034_antes = (df["device_code"] == "KPCL0034").sum()

df = df.loc[~es_dup_par].reset_index(drop=True)

n_abril_despues = (df["device_id"] == "9510a455-b0e9-4932-8be1-03976d31228a").sum()
n_kpcl0034_despues = (df["device_code"] == "KPCL0034").sum()
caida_pct = (1 - n_abril_despues / n_abril_antes) * 100
print(f"Filas abril (9510a455) antes: {n_abril_antes:,}")
print(f"Filas abril (9510a455) despues: {n_abril_despues:,}")
print(f"Caida: {caida_pct:.1f}% (esperado: ~49.6% -- si es ~24.8% hay un bug de indexacion)")

# Sanity check: la caida total de KPCL0034 (unificado) debe cerrar con
# "49.6% de abril eliminado" x "fraccion de KPCL0034 que era abril".
frac_abril = n_abril_antes / n_kpcl0034_antes
caida_total_esperada = (caida_pct / 100) * frac_abril * 100
caida_total_real = (1 - n_kpcl0034_despues / n_kpcl0034_antes) * 100
print(f"Fraccion de KPCL0034 que era abril (antes del dedup): {frac_abril * 100:.1f}%")
print(f"Caida total esperada en KPCL0034: {caida_total_esperada:.1f}%")
print(f"Caida total real en KPCL0034: {caida_total_real:.1f}%")

# Recalcular delta_peso/delta_t desde cero sobre el frame YA deduplicado --
# los valores de arriba quedaron obsoletos (indices/vecinos cambiaron).
df["delta_peso"] = df.groupby("device_id", observed=True)["peso"].diff()
df["delta_t"] = df.groupby("device_id", observed=True)["ts"].diff().dt.total_seconds()
df["abs_delta_peso"] = df["delta_peso"].abs()
df["delta_peso_valido"] = df["delta_peso"].where(df["delta_t"] <= GAP_CUTOFF_S)

# --- Nivel de peso por periodo (UUID), post-dedup -----------------------------
# El corte de cadencia es la variable relevante, no el UUID en si -- pero acá
# coinciden exactamente (cada UUID = un periodo de cadencia distinto). Objetivo:
# saber si abril y mayo+ tienen nivel de peso base distinto MAS ALLA del sesgo
# de representacion que el dedup acaba de corregir.
print("--- Nivel de peso por periodo (UUID), post-dedup ---")
for _periodo, _uuid in [
    ("abril", "9510a455-b0e9-4932-8be1-03976d31228a"),
    ("mayo-agosto", "3a460074-e7c3-41bf-ae5a-a011445f927a"),
]:
    _g = df.loc[df["device_id"] == _uuid, "peso"]
    _mediana_g = _g.median()
    print(f"{_periodo}: mediana={_mediana_g:.1f}g  MAD={(_g - _mediana_g).abs().median():.1f}g  n={len(_g):,}")

print()
print(f"Lecturas totales (despues de dedup): {len(df):,}")
print(df["device_code"].value_counts())
print(f"Memoria del DataFrame: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

# --- Guardar cache para los notebooks 2/3/4 (evita repetir el parseo de
# los 312MB crudos + el dedup en cada uno) -----------------------------------
DATA_DIR = NOTEBOOK_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)
CACHE_CSV = DATA_DIR / "lecturas_limpias.csv"
df[["device_id", "device_code", "ts", "peso"]].to_csv(CACHE_CSV, index=False)
print(f"Cache escrito: {CACHE_CSV} ({CACHE_CSV.stat().st_size / 1e6:.1f} MB)")


## Hallazgo documentado - duplicado de escritura en abril 2026 (KPCL0034)

**Fecha del hallazgo:** 2026-08-29, durante este mismo Paso 1.

**Que se encontro:** el UUID de abril de KPCL0034 (`9510a455-b0e9-4932-8be1-03976d31228a`)
tiene, para el **49.6%** de sus filas en `readings.csv`, un segundo registro casi
instantaneo (~0.2-0.25s despues) del mismo evento fisico, con **el mismo peso**
en el **99.91%** de esos pares. Patron tipico observado: `80.0g` a las
`02:48:43.398695`, y de nuevo `80.0g` a las `02:48:43.636000` (0.237s despues) --
cada ~30s, repetido durante todo abril.

**Por que no es senial real:** es la propia `readings.csv` escribiendo cada
muestra 2 veces durante ese periodo -- no involucra a `readings_rows.csv` (esa
tabla no tiene filas de este UUID). El UUID de mayo-en-adelante y KPCL0035 NO
tienen este patron (fraccion de intervalos <1s ahi: ~0.006%, esencialmente cero).

**Por que importa:** este mismo tipo de heterogeneidad (cadencia/duplicados
distintos entre abril y mayo-jun) ya le costo al Ciclo Alpha v1 una caida de F1
de 0.16 puntos en su momento (Exp 08, documentado en `av1_EXPERIMENTOS_DETALLE.md`)
cuando se mezclo sin corregir. Sin deduplicar, cualquier metrica basada en
`delta_t`/velocidad/duracion de estabilidad para KPCL0034 queda sesgada hacia
el regimen de abril, que domina en cantidad de filas duplicadas.

**Decision tomada:** deduplicar de forma estricta y mecanica -- se elimina la
segunda fila de cada par cuando `delta_t < 1s` **y** `delta_peso == 0` exacto
(se conserva la primera). Los pares con `delta_t < 1s` pero peso **distinto**
se dejan intactos, tal cual, sin deduplicar -- podrian ser un evento real que
coincidio justo con el instante del duplicado, y el criterio de dedup debe ser
estricto (delta_t + delta_peso==0 exacto), no por cercania temporal sola.

**Verificacion esperada:** la caida de filas del UUID de abril tras el dedup
debe ser ~49.6% (no ~24.8%, que indicaria estar eliminando ambos miembros de
algunos pares por error de indexacion tras el `groupby`).


In [ ]:
# === 2. NIVEL ==================================================================
# Gramos de comida (KPCL0034) y gramos/mL de agua (KPCL0035) NO son la misma
# magnitud fisica -- todo el bloque va por device_code, sin version global.
nivel_por_device = df.groupby("device_code", observed=True)["peso"].agg(
    n="count",
    mediana="median",
    std="std",
    mad=lambda s: (s - s.median()).abs().median(),
)
print("--- Nivel por device_code ---")
print(nivel_por_device.round(2))

pct_peso = [1, 5, 10, 25, 50, 75, 90, 95, 99]
for _code, _grupo in df.groupby("device_code", observed=True):
    _dist = _grupo["peso"].quantile([p / 100 for p in pct_peso])
    _dist.index = [f"P{p:02d}" for p in pct_peso]
    print()
    print(f"--- Percentiles de peso ({_code}, g) ---")
    print(_dist.round(2))

# === 3. RESOLUCION =============================================================
peso_valido = df["peso"].dropna()
frac_entero = (peso_valido % 1 == 0).mean()
print(f"Lecturas de peso con valor entero exacto (todos los dispositivos): {frac_entero * 100:.2f}%")
print()
print("Valores unicos de peso por device_code:")
print(df.groupby("device_code", observed=True)["peso"].nunique())

for _code, _grupo in df.groupby("device_code", observed=True):
    print()
    print(f"--- 10 valores de peso mas frecuentes ({_code}) ---")
    print(_grupo["peso"].value_counts().head(10))

# === 4. DINAMICA (delta_peso) ==================================================
pct_delta = [50, 90, 95, 99, 99.5, 99.9]
for _code, _grupo in df.groupby("device_code", observed=True):
    _dist = _grupo["abs_delta_peso"].quantile([p / 100 for p in pct_delta])
    _dist.index = [f"P{p}" for p in pct_delta]
    print()
    print(f"--- Percentiles de |delta_peso| ({_code}, g) ---")
    print(_dist.round(3))

# Direccion del cambio -- subir no es lo mismo que bajar
for _code, _grupo in df.groupby("device_code", observed=True):
    _subidas = _grupo.loc[_grupo["delta_peso"] > 0, "delta_peso"]
    _bajadas = _grupo.loc[_grupo["delta_peso"] < 0, "delta_peso"].abs()
    print()
    print(f"--- {_code}: subidas n={len(_subidas):,}, bajadas n={len(_bajadas):,} ---")
    print("Subidas:", _subidas.quantile([0.50, 0.90, 0.95, 0.99]).round(3).to_dict())
    print("Bajadas:", _bajadas.quantile([0.50, 0.90, 0.95, 0.99]).round(3).to_dict())

frac_cero_por_device = df.groupby("device_code", observed=True)["delta_peso"].apply(lambda s: (s.dropna() == 0).mean())
print()
print("--- % lecturas con delta_peso == 0, por device_code ---")
print((frac_cero_por_device * 100).round(2))

# === 5. TEMPORAL (delta_t, gaps) ===============================================
for _code, _grupo in df.groupby("device_code", observed=True):
    _dt = _grupo["delta_t"]
    print()
    print(f"--- delta_t ({_code}, s) ---")
    print(f"media: {_dt.mean():.2f}   mediana: {_dt.median():.2f}")
    print(_dt.quantile([0.90, 0.95]).round(2))

# Abril vs mayo-en-adelante DENTRO de KPCL0034, ya deduplicado -- confirma si
# el dedup dejo a ambos periodos con cadencia comparable.
for _dev, _grupo in df[df["device_code"] == "KPCL0034"].groupby("device_id", observed=True):
    _dt = _grupo["delta_t"]
    print()
    print(f"--- delta_t (KPCL0034, UUID {_dev[:8]}..., post-dedup, s) ---")
    print(f"media: {_dt.mean():.2f}   mediana: {_dt.median():.2f}")

is_gap = df["delta_t"] > GAP_CUTOFF_S
n_gaps_por_device = is_gap.fillna(False).groupby(df["device_code"], observed=True).sum()
print()
print(f"--- Gaps (> {GAP_CUTOFF_S}s) por device_code ---")
print(n_gaps_por_device)


In [ ]:
# === 6. VELOCIDAD (delta_peso / delta_t) ======================================
df["velocidad_peso"] = df["delta_peso_valido"] / df["delta_t"]
pct_vel = [50, 90, 95, 99, 99.5, 99.9]
for _code, _grupo in df.groupby("device_code", observed=True):
    _vel_abs = _grupo["velocidad_peso"].abs().dropna()
    _dist = _vel_abs.quantile([p / 100 for p in pct_vel])
    _dist.index = [f"P{p}" for p in pct_vel]
    print()
    print(f"--- Percentiles de |velocidad_peso| ({_code}, g/s) ---")
    print(_dist.round(4))

# === 7. PERSISTENCIA (duracion de estabilidad) ================================
# ponytail: estabilidad TOLERANTE (variacion chica pero != 0) sigue pendiente
# a proposito -- recien con la distribucion de |delta_peso| (bloque 4) se
# puede elegir un umbral de tolerancia justificado, no inventado antes de tiempo.
paso_estable = (df["delta_peso"] == 0) & (~is_gap.fillna(False))
corrida_id = (~paso_estable).groupby(df["device_code"], observed=True).cumsum()
duraciones = df["delta_t"][paso_estable].groupby(
    [df.loc[paso_estable, "device_code"], corrida_id[paso_estable]], observed=True,
).sum()

for _code in df["device_code"].cat.categories:
    _dur_code = duraciones.loc[_code] if _code in duraciones.index.get_level_values(0) else pd.Series(dtype=float)
    print()
    print(f"--- Duracion de corridas estables ({_code}) ---")
    if len(_dur_code) > 0:
        print(f"Corridas encontradas: {len(_dur_code):,}")
        print(_dur_code.quantile([0.50, 0.90, 0.95, 0.99]).round(1))
        print(f"maxima: {_dur_code.max():.1f} s ({_dur_code.max() / 60:.1f} min)")
    else:
        print("Sin corridas estables encontradas.")

# === 8. COMPORTAMIENTO TEMPORAL ================================================
for _dev, _grupo in df.groupby("device_id", observed=True):
    _code = UUID_TO_CODE[_dev]
    _autocorr = _grupo["peso"].autocorr(lag=1)
    print(f"Autocorrelacion lag-1 ({_code}, UUID {_dev[:8]}...): {_autocorr:.4f}")

df["hora_stgo"] = df["ts"].dt.tz_convert("America/Santiago").dt.hour
for _code, _grupo in df.groupby("device_code", observed=True):
    _por_hora = _grupo.groupby("hora_stgo")["peso"].agg(
        mediana="median", mad=lambda s: (s - s.median()).abs().median(),
    )
    print()
    print(f"--- Peso por hora del dia, Santiago ({_code}) ---")
    print(_por_hora.round(2))

# === Resumen - Paso 1 (una fila por device_code) ==============================
filas_resumen = []
for _code, _grupo in df.groupby("device_code", observed=True):
    _mediana = _grupo["peso"].median()
    _std = _grupo["peso"].std()
    _mad = (_grupo["peso"] - _mediana).abs().median()
    _abs_delta = _grupo["abs_delta_peso"].dropna()
    _frac_cero_g = (_grupo["delta_peso"].dropna() == 0).mean()
    _dt_g = _grupo["delta_t"]
    _vel_g = _grupo["velocidad_peso"].abs().dropna()
    _dur_g = duraciones.loc[_code] if _code in duraciones.index.get_level_values(0) else pd.Series(dtype=float)
    filas_resumen.append({
        "device_code": _code,
        "n": len(_grupo),
        "mediana_peso_g": round(_mediana, 2),
        "std_peso_g": round(_std, 2),
        "mad_peso_g": round(_mad, 2),
        "valores_unicos": int(_grupo["peso"].nunique()),
        "abs_delta_p50_g": round(_abs_delta.quantile(0.50), 3) if len(_abs_delta) else None,
        "abs_delta_p99_g": round(_abs_delta.quantile(0.99), 3) if len(_abs_delta) else None,
        "pct_delta_cero": round(_frac_cero_g * 100, 2),
        "delta_t_mediana_s": round(_dt_g.median(), 2),
        "gaps_5min": int(n_gaps_por_device.get(_code, 0)),
        # P95 no sirve aca: con pct_delta_cero ~98%, el P95 de cualquier
        # magnitud derivada de delta_peso cae dentro de la masa de ceros.
        # P99 recien empieza a asomar senial real; P99.9 es donde probablemente
        # este el rango de "esto si es un evento".
        "velocidad_p99_g_s": round(_vel_g.quantile(0.99), 4) if len(_vel_g) else None,
        "velocidad_p999_g_s": round(_vel_g.quantile(0.999), 4) if len(_vel_g) else None,
        "duracion_estable_p50_s": round(_dur_g.quantile(0.50), 1) if len(_dur_g) else None,
    })
resumen = pd.DataFrame(filas_resumen).set_index("device_code")
resumen


## Validacion final - abril vs. mayo-agosto son homogeneos?

"No hay diferencia de tara/hardware" y "el fondo es comparable sin ajuste" son
afirmaciones distintas -- la segunda no se sigue automaticamente de la primera
(ya sabiamos, de los bloques 1-7, que MAD y cadencia difieren entre periodos).
El siguiente bloque de codigo cierra ambas preguntas por separado, con
evidencia, en vez de asumir una a partir de la otra.


In [ ]:
# === VALIDACION 1: nivel absoluto -- valle (P05) vs pico (P95) semanal ========
# El pico depende de cuanta comida sirvieron (variable operacional, nada que
# ver con tara). El valle (plato vacio) SI deberia ser una constante fisica si
# la tara/calibracion no cambio entre periodos.
_g = df.set_index("ts").groupby("device_id", observed=True)["peso"]
_p05 = _g.resample("7D").quantile(0.05)
_p95 = _g.resample("7D").quantile(0.95)
for _uuid, _periodo in [
    ("9510a455-b0e9-4932-8be1-03976d31228a", "abril"),
    ("3a460074-e7c3-41bf-ae5a-a011445f927a", "mayo-agosto"),
]:
    _p05_s = _p05.loc[_uuid].dropna()
    _p95_s = _p95.loc[_uuid].dropna()
    print(f"{_periodo}: P05 semanal mediana={_p05_s.median():.1f}g (rango {_p05_s.min():.0f}-{_p05_s.max():.0f})  "
          f"P95 semanal mediana={_p95_s.median():.1f}g (rango {_p95_s.min():.0f}-{_p95_s.max():.0f})")

print("Conclusion: el valle difiere poco entre periodos (~8g) vs. el pico (~26g)")
print("-> el offset de nivel es apetito/servido, NO tara/hardware.")

# === VALIDACION 2: dinamica de Δpeso/velocidad/estabilidad por periodo ========
# Esta es la pregunta que realmente importa para el baseline: no el nivel,
# sino si la FORMA de la variacion (lo que alimenta al detector) es comparable.
for _uuid, _periodo in [
    ("9510a455-b0e9-4932-8be1-03976d31228a", "abril (post-dedup)"),
    ("3a460074-e7c3-41bf-ae5a-a011445f927a", "mayo-agosto"),
]:
    _grupo = df[df["device_id"] == _uuid]
    print()
    print(f"--- {_periodo} ---")
    print("|delta_peso| P50/90/95/99/99.5/99.9:",
          _grupo["abs_delta_peso"].quantile([.5, .9, .95, .99, .995, .999]).round(3).tolist())
    print("velocidad P50/90/95/99/99.5/99.9:",
          _grupo["velocidad_peso"].abs().quantile([.5, .9, .95, .99, .995, .999]).round(4).tolist())
    _dur = duraciones.loc[_uuid] if _uuid in duraciones.index.get_level_values(0) else pd.Series(dtype=float)
    if len(_dur) == 0:
        # duraciones esta indexado por device_code, no device_id -- fallback
        _mask_periodo = df["device_id"] == _uuid
        _dur = df.loc[_mask_periodo & paso_estable, "delta_t"].groupby(
            corrida_id[_mask_periodo & paso_estable], observed=True,
        ).sum()
    print(f"duracion estable P50/P90/P95: {_dur.quantile([.5, .9, .95]).round(1).tolist()}  (n corridas: {len(_dur):,})")

print()
print("Conclusion: cuerpo de la distribucion (hasta P99) comparable entre")
print("periodos. Cola extrema (P99.9) y duracion estable en P90/P95 NO lo son")
print("-- abril (cadencia ~15s) acumula corridas estables ~2x mas largas que")
print("mayo+ (30s) en esos percentiles altos: artefacto residual de cadencia")
print("que el dedup de mas arriba NO corrige (igualo la mediana de cadencia,")
print("no cuantas lecturas hacen falta para que un gap corte una corrida).")


## Cierre del Paso 1

| Pregunta | Resultado |
|---|---|
| Nivel absoluto de peso -- tara/hardware distinto entre periodos? | **No.** Valle (P05) casi igual entre abril y mayo-agosto (~8g); el pico (P95, ~26g de diferencia) es apetito/servido, variable operacional. |
| Cuerpo de la dinamica (`|delta_peso|`/velocidad hasta P99) comparable? | **Si.** P50-P99 practicamente identicos entre abril-post-dedup y mayo-agosto. |
| Cola extrema (P99.9) y duracion de estabilidad (P90/P95) comparables? | **No.** Abril acumula corridas estables ~2x mas largas en P90/P95 -- artefacto residual de cadencia (~15s vs 30s) que el dedup de duplicados NO corrige. |

**Decision para el Paso 2:** el baseline de nivel/dinamica central puede tratar
a KPCL0034 como un solo dispositivo logico (`device_code`), tal como se hizo en
todo este notebook. Pero cualquier regla que dependa de **cuanto dura
normalmente estar quieto** (umbrales de duracion de estabilidad, gap interno,
etc.) debe tratarse por periodo de cadencia, o normalizarse explicitamente por
cadencia antes de fijar un umbral unico -- de lo contrario el umbral queda
sesgado hacia cualquiera de los dos periodos que domine en cantidad de datos.


## Resumen final (tabla y gráfico)

Consolida en una sola tabla y un solo gráfico los tres hallazgos de este
notebook: el impacto del dedup de abril, la comparación valle/pico por
período (Validación 1), y la duración de estabilidad por período
(Validación 2) — reutilizando las variables ya calculadas arriba, sin
recargar ni recalcular la señal completa.


In [ ]:
import matplotlib.pyplot as plt

# --- Tabla resumen: reutiliza _p05/_p95 (Validacion 1) y paso_estable/corrida_id (bloque 7) ---
_periodos_resumen = [
    ("KPCL0034 - abril", "9510a455-b0e9-4932-8be1-03976d31228a"),
    ("KPCL0034 - mayo-agosto", "3a460074-e7c3-41bf-ae5a-a011445f927a"),
    ("KPCL0035", "0dc601c0-1533-40c5-b606-6d89eb2d4042"),
]

filas_resumen = []
for _nombre, _uuid in _periodos_resumen:
    _mask = df["device_id"] == _uuid
    _p05_s = _p05.loc[_uuid].dropna()
    _p95_s = _p95.loc[_uuid].dropna()
    _dur = df.loc[_mask & paso_estable, "delta_t"].groupby(
        corrida_id[_mask & paso_estable], observed=True,
    ).sum()
    filas_resumen.append({
        "periodo": _nombre,
        "valle_p05_g": round(_p05_s.median(), 1),
        "pico_p95_g": round(_p95_s.median(), 1),
        "abs_delta_p99_g": round(df.loc[_mask, "abs_delta_peso"].quantile(0.99), 3),
        "duracion_estable_p95_s": round(_dur.quantile(0.95), 1) if len(_dur) else float("nan"),
    })

resumen_final_df = pd.DataFrame(filas_resumen).set_index("periodo")
print("--- Resumen final Paso 1 ---")
print(resumen_final_df)

# --- Grafico resumen: 3 paneles ------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: impacto del dedup de abril
axes[0].bar(["Caida UUID abril", "Caida total KPCL0034"], [caida_pct, caida_total_real],
            color=["#c0392b", "#e67e22"])
axes[0].set_ylabel("% de filas eliminadas")
axes[0].set_title("Impacto del dedup de abril")
for _i, _v in enumerate([caida_pct, caida_total_real]):
    axes[0].text(_i, _v + 1, f"{_v:.1f}%", ha="center")

# Panel 2: valle vs pico por periodo
_x = np.arange(len(resumen_final_df))
axes[1].bar(_x - 0.2, resumen_final_df["valle_p05_g"], width=0.4, label="Valle (P05)", color="#2980b9")
axes[1].bar(_x + 0.2, resumen_final_df["pico_p95_g"], width=0.4, label="Pico (P95)", color="#27ae60")
axes[1].set_xticks(_x)
axes[1].set_xticklabels(resumen_final_df.index, rotation=20, ha="right")
axes[1].set_ylabel("Peso semanal (g)")
axes[1].set_title("Valle vs pico por periodo")
axes[1].legend()

# Panel 3: duracion de estabilidad P95 por periodo
axes[2].bar(resumen_final_df.index, resumen_final_df["duracion_estable_p95_s"], color="#8e44ad")
axes[2].set_xticks(np.arange(len(resumen_final_df)))
axes[2].set_xticklabels(resumen_final_df.index, rotation=20, ha="right")
axes[2].set_ylabel("Duracion estable P95 (s)")
axes[2].set_title("Duracion de estabilidad por periodo")

fig.suptitle("Paso 1 - Resumen: dedup, nivel y duracion por periodo")
fig.tight_layout()
plt.show()
